In [6]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

# --- PASO 1: CONFIGURACIÓN DEL DRIVER (EL CUERPO) ---
# Se define al principio para evitar abrir múltiples navegadores innecesariamente
options = Options()
options.add_argument("--headless") # Ejecución invisible en el servidor Docker
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)

# --- PASO 2: INICIALIZACIÓN DE VARIABLES (LA MEMORIA) ---
# La lista debe estar FUERA del bucle para acumular los datos de todas las páginas
datos_totales = []
paginas_objetivo = 5 # Límite para el ejercicio de laboratorio

try:
    # URL inicial del caso de estudio (E-commerce, Inmobiliario, etc.)
    # RECUERDA: Cambia esta URL por la que vas a scrapear realmente
    driver.get("https://books.toscrape.com/")

    # --- PASO 3: BUCLE DE PAGINACIÓN ---
    for i in range(paginas_objetivo):
        print(f"--- Procesando Página {i + 1} ---")

        # Espera explícita para asegurar que los elementos carguen
        WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.CLASS_NAME, "product_pod"))
        )

        # CAPTURA: Extraer elementos de la vista actual
        productos = driver.find_elements(By.CLASS_NAME, "product_pod")
        for p in productos:
            datos_totales.append(p.text) # Guardamos en nuestra "mochila"

        # NAVEGACIÓN: Lógica para saltar a la siguiente página
        try:
            # Buscamos el botón "Siguiente" usando XPATH
            boton_siguiente = driver.find_element(By.XPATH, "next")
            
            # Hacemos clic y esperamos la carga de la nueva interfaz
            boton_siguiente.click()
            time.sleep(3) # Pausa estratégica para el renderizado de JavaScript

        except Exception:
            print("Se ha llegado a la última página o el botón no es visible.")
            break # Detiene el bucle si ya no hay más páginas

    print(f"\nProceso finalizado con éxito.")
    print(f"Total de registros capturados: {len(datos_totales)}")

except Exception as e:
    print(f"Error detectado durante la ejecución: {e}")

finally:
    # --- PASO 4: CIERRE SEGURO (CRUCIAL PARA DOCKER) ---
    # Esto evita que los procesos de Chrome 'zombis' agoten la RAM
    driver.quit()

--- Procesando Página 1 ---
Se ha llegado a la última página o el botón no es visible.

Proceso finalizado con éxito.
Total de registros capturados: 20
